In [6]:
"""
레시피 예상 비용 계산 스크립트
- recipe_ingredients의 amount + unit_id 기반으로 재료별 비용 계산
- recipes.estimated_cost 업데이트
"""

import mysql.connector
from mysql.connector import Error
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("calculate_cost.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "database": "cooking_db",
    "user": "root",
    "password": "root",
    "charset": "utf8mb4"
}


def ensure_cost_column(cursor):
    """recipes 테이블에 estimated_cost 컬럼 없으면 추가"""
    cursor.execute("SHOW COLUMNS FROM recipes LIKE 'estimated_cost'")
    if not cursor.fetchone():
        cursor.execute("ALTER TABLE recipes ADD COLUMN estimated_cost DECIMAL(10,2) DEFAULT NULL")
        log.info("recipes.estimated_cost 컬럼 추가")


def calculate_costs(cursor):
    # 1. 단위 변환 테이블 로드
    cursor.execute("SELECT id, code, to_base, base_code FROM units")
    units = {row['id']: row for row in cursor.fetchall()}

    # 2. 재료 가격 로드 (avg_price = 100g 또는 100ml 당 가격)
    cursor.execute("SELECT id, name, avg_price FROM ingredients WHERE avg_price IS NOT NULL")
    ingredients = {row['id']: row for row in cursor.fetchall()}

    # 3. 전체 레시피 목록
    cursor.execute("SELECT id, title FROM recipes")
    recipes = cursor.fetchall()
    log.info(f"레시피 {len(recipes)}개 처리 시작")

    updated = 0
    skipped = 0

    for recipe in recipes:
        recipe_id = recipe['id']

        # 4. 해당 레시피 재료 목록
        cursor.execute("""
            SELECT ri.ingredient_id, ri.amount, ri.unit_id
            FROM recipe_ingredients ri
            WHERE ri.recipe_id = %s AND ri.ingredient_id IS NOT NULL
        """, (recipe_id,))
        items = cursor.fetchall()

        total_cost = 0.0
        has_price = False
        missing = []

        for item in items:
            ing_id  = item['ingredient_id']
            amount  = item['amount']
            unit_id = item['unit_id']

            # amount나 unit_id 없으면 스킵
            if not amount or not unit_id:
                continue

            ing = ingredients.get(ing_id)
            if not ing:
                missing.append(ing_id)
                continue

            unit = units.get(unit_id)
            if not unit:
                continue

            avg_price = float(ing['avg_price'])
            to_base   = float(unit['to_base'])
            base_code = str(unit['base_code'])

            # 기본단위로 변환된 양
            base_amount = float(amount) * to_base

            # 비용 계산
            if base_code in ('g', 'ml'):
                # 100g/100ml 당 가격 기준
                cost = (avg_price / 100.0) * base_amount
            elif base_code == 'piece':
                # 개수 기준 (avg_price = 1개당 가격)
                cost = avg_price * base_amount
            else:
                continue

            total_cost += cost
            has_price = True

        if has_price and total_cost > 0:
            cursor.execute(
                "UPDATE recipes SET estimated_cost = %s WHERE id = %s",
                (round(total_cost, 2), recipe_id)
            )
            updated += 1
        else:
            skipped += 1

    return updated, skipped


def main():
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)
        log.info("DB 연결 성공")
    except Error as e:
        log.error(f"DB 연결 실패: {e}")
        return

    try:
        ensure_cost_column(cursor)
        conn.commit()

        updated, skipped = calculate_costs(cursor)
        conn.commit()

        log.info(f"=== 완료 ===")
        log.info(f"비용 계산 완료: {updated}개 / 스킵: {skipped}개")

        # 결과 샘플 출력
        cursor.execute("""
            SELECT title, estimated_cost
            FROM recipes
            WHERE estimated_cost IS NOT NULL
            ORDER BY estimated_cost ASC
            LIMIT 20
        """)
        rows = cursor.fetchall()
        log.info("=== 비용 낮은 레시피 TOP 20 ===")
        for r in rows:
            log.info(f"  {r['title']:30s} {r['estimated_cost']:>10,.0f}원")

    except Error as e:
        log.error(f"DB 오류: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    main()

2026-03-04 14:42:14,840 [INFO] DB 연결 성공


2026-03-04 14:42:14,857 [INFO] 레시피 333개 처리 시작
2026-03-04 14:42:15,399 [INFO] === 완료 ===
2026-03-04 14:42:15,402 [INFO] 비용 계산 완료: 242개 / 스킵: 91개
2026-03-04 14:42:15,405 [INFO] === 비용 낮은 레시피 TOP 20 ===
2026-03-04 14:42:15,407 [INFO]   하면 할수록 빠져든다... 집에서 빚는 우리 술 우리 막걸리          2원
2026-03-04 14:42:15,408 [INFO]   손님이 줄어든 이유? 다 흑백요리사 때문이에요              11원
2026-03-04 14:42:15,410 [INFO]   폭신~폭신한 에그 샌드위치 Soft and fluffy egg sandwich         34원
2026-03-04 14:42:15,413 [INFO]   더도 말고 덜도 말고 딱 6분 컷! 왕가래떡 떡볶이           34원
2026-03-04 14:42:15,415 [INFO]   콩 없이 만드는 콩국수! 2021ver.                 34원
2026-03-04 14:42:15,417 [INFO]   초간단 1분 라볶이                             34원
2026-03-04 14:42:15,419 [INFO]   토마토 주스로 스튜를 만든다고요? 집에 있는 재료만으로 도전!🍅         43원
2026-03-04 14:42:15,421 [INFO]   '시간대별 계란 삶기' (여러분의 취향은 몇 분?)           43원
2026-03-04 14:42:15,422 [INFO]   김밥의 수많은 속재료가 부담된다면? '이것'만 준비하세요, 간단하게 김밥 맛집 흉내내기 가능!         68원
2026-03-04 14:42:15,423 [INFO]   겉절이 어렵지 않아요! 만능양념장과 함께라면 봄나물 완전정복~      